# OPS Environment Tutorial

This notebook contains the tutorial for the 2025 IEEE BioCAS Grand Challenge - Closed-loop Neural Decoding for Motor Control of non-Human Primates.

For this Grand Challenge, we are performing a closed-loop simulation of a brain model (from hereon referred to as an OPS (Online Prosthesis Simulator)). An example of a model making prediction can be seen below:

![Alt Text](closed_loop_trajectory.gif)

In this tutorial, we will go over how the OPS can be setup and used in a closed-loop environment, while using **NeuroBench** to evaluate the performance of the trained model.

## Import packages

For this notebook, you will need to install the NeuroBench version used for this Grand Challenge.
You can install the package with the following instruction:

``` bash
pip install git+https://github.com/NeuroBench/neurobench.git@2025_GC
```

For other packages, please refer to the official NeuroBench for version compatibility (The installation process should include all of the dependeded packages).

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import os
import sys

from neurobench.envs import OPS, OPSEnv
from neurobench.models.torch_model import TorchModel
from neurobench.benchmarks import BenchmarkClosedLoop

from neurobench.metrics.workload import (
    ActivationSparsity,
    SynapticOperations,
)
from neurobench.metrics.static import (
    Footprint,
    ConnectionSparsity,
)

ModuleNotFoundError: No module named 'torch'

## Global Variable Definition

Here we define the where the model's weight and the OPS neurons is stored, along with some variable used for the closed-loop simulation

In [ ]:
file_path = os.path.dirname(os.path.abspath(__file__))
model_path = os.path.join(file_path, "OPS_model_state_dict.pth")
neuron_path = os.path.join(file_path, "neuron1.csv")

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

time_step = 0.01
num_neurons=96
max_duration = 3.0
min_time_in_target = 0.5 # value in seconds
target_size = 2.5

## Declare OPS and OPS Environment

The OPS is the brain behind our closed-loop simulation.

To initialise the OPS, we can define it according to the cell below.

The OPS class contains the following paramters:

- num_neurons: Number of neurons used by the OPS.
- time_step: The time taken per iteration (s).
- upper_lmax: upper limit of the maximum firing rate of a neuron
- lower_lmax: lower limit of the maximum firing rate of a neuron
- upper_lmin: upper limit of the minimum firing rate of a neuron
- zero_prob: probability of the neuron producing no spikes
- device: device to run the OPS on (cpu or cuda)

In [ ]:
ops = OPS(
    num_neurons=num_neurons,
    time_step=time_step,
    upper_lmax=100,
    lower_lmax=40,
    upper_lmin=5,
    zero_prob=0.5,
    device=device
)

For this Grand Challenge, the paramters of the neurons in the OPS is already initialised for you. You can load them with the following line of code.

Please remember the file loaded in this example is not a part of the Grand Challenge.

In [ ]:
ops.assign_neurons(neuron_path)

Now that the OPS is initialised with the prechosen neuron paramters, we can initialise the closed-loop environment.

The **OPSEnv** is defined with the following paramters:

- ops: The OPS object that governs how the state updates.
- max_duration: The maximum duration of the simulation (s).
- min_time_in_target: min time for the model output to stay in target area
- side_radius: radius of the OPS environment.
- min_distance: minimum distance to the target from the origin.
- target_size: size of the target area.
- device: device to run the OPS on (cpu or cuda)

In [ ]:
env = OPSEnv(
    ops=ops,
    max_duration=max_duration,
    min_time_in_target=min_time_in_target,
    side_radius=10,
    min_distance=8,
    target_size=target_size,
    device=device
)

## Model Definition

Below is a simple fully-connected neural network using PyTorch.

For your model to be evaluated with NeuroBench, you need to make the following changes to your model:

1. To account for the preprocessing performed by your model, you need to define a register_buffer that keep track of the incoming observation to your model.
    For example, if your model's forward propagation requires the previous five observation as well as the current observation, your register_buffer will concatenate the current observation and then pop off the oldest (out of scope) observation, like the following:

    ``` python
        def __init__(self, ...):
            ...

            self.register_buffer("data_buffer", torch.zeros(1, input_dim).type(torch.float32), persistent=False)

            ...
        
        def forward(self, x):
            self.data_buffer = torch.cat((self.data_buffer, x), dim=0)
            self.data_buffer = self.data_buffer[1:, :]
            
            ...

            return pred
    ```

    The ``` self.data_buffer ``` variable allows the NeuroBench harness to calculate the footprint used by the model is accurate in a real world scenario, as real world applications often uses additional memory for input data processing.

In [ ]:
class ANNModel(nn.Module):
    def __init__(
        self, 
        input_dim, 
        layer1=16, 
        layer2=32, 
        output_dim=2,
        bin_window=0.2, 
        sampling_rate=0.004, 
        drop_rate=0.5
    ):
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.layer1 = layer1
        self.layer2 = layer2
        self.drop_rate = drop_rate

        self.bin_window_time = bin_window
        self.sampling_rate = sampling_rate
        self.bin_window_size = int(self.bin_window_time / self.sampling_rate)

        self.fc1 = nn.Linear(self.input_dim, self.layer1)
        self.fc2 = nn.Linear(self.layer1, self.layer2)
        self.fc3 = nn.Linear(self.layer2, self.output_dim)
        self.dropout = nn.Dropout(self.drop_rate)
        self.batchnorm1 = nn.BatchNorm1d(self.layer1)
        self.batchnorm2 = nn.BatchNorm1d(self.layer2)
        self.activation = nn.ReLU()
        self.batch_size = 1

        self.register_buffer("data_buffer", torch.zeros(1, 1, input_dim).type(torch.float32), persistent=False)

    def forward(self, x):
        self.data_buffer = torch.cat((self.data_buffer, x), dim=0)
        self.data_buffer = self.data_buffer[1:, :, :]

        x = self.activation(self.fc1(x.view(self.batch_size, -1)))
        x = self.activation(self.dropout(self.fc2(x)))
        x = self.fc3(x)

        pred = x.squeeze(dim=0)

        return pred

We can initialise the model with standard PyTorch model initialisation call, then load the model's weight.

After this is done, we will pass the model into NeuroBench's model wrapper. In this example, we are using the TorchModel() class as the wrapper. If you're using SNNTorch, you can also use the SNNTorchModel() as the wrapper.

In [ ]:
net = ANNModel(input_dim=num_neurons)
net.load_state_dict(torch.load(model_path, map_location=device))
model = TorchModel(net)

## Benchmarking your model

To benchmark your model, you first need to define what metrics your want to use.

There are two types of metrics, the static metrics (metrics that are only dependent on the model itself) and workload metrics (metrics for when the model is undergoing inference).

In this Grand Challenge, we are evaluating the following metrics:

1. Static Metrics: model's memory footprint and connection sparsity (sparsity of the model's weight).

2. Workload Metrics: model's activation sparsity and synaptic operations.

In [ ]:
static_metrics = [Footprint, ConnectionSparsity]
workload_metrics = [ActivationSparsity, SynapticOperations]

You can initialise the closed-loop benchmark ( BenchmarkClosedLoop() ) with the following line of code.

The BenchmarkClosedLoop() class have the following parameters:

- agent: A NeuroBenchAgent (SNNTorchAgent/TorchModel...).
- environment: A Gym environment. In this case it is the OPSEnv.
- preprocessors: A list of NeuroBenchPreProcessors or callable functions (e.g. lambda) with matching interfaces. If not used, just pass in an empty list.
- postprocessors: A list of NeuroBenchPostProcessors or callable functions (e.g. lambda) with matching interfaces. If not used, just pass in an empty list.
- metric_list: A list of lists of StaticMetric and WorkloadMetric classes of metrics to run.
    First item is StaticMetrics, second item is WorkloadMetrics.

In [ ]:
benchmark = BenchmarkClosedLoop(model, env, [], [], [static_metrics, workload_metrics])

Once the benchmark is initialised, we can just call the run method on the benchmark object to execute the closed-loop benchmark.

The run method expects the following input parameters:

- nr_interactions: Number of interactions with the environment.
- Maximum length of an interaction with the environment.

At the end of the benchmark, you will be informed of the number of successful trials and a dictionary with all of the metrics will be returned by the run() method.

In [ ]:
results = benchmark.run(nr_interactions=50, max_length=300, device=device)
print(results)